In [ ]:
# -------------------------
# 1. Imports
# -------------------------
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import faiss
import numpy as np

# -------------------------
# 2. Sample Data
# -------------------------
documents = [
    # --- Geography ---
    Document(page_content="The Eiffel Tower was built in 1889 in Paris, France and is one of the most visited landmarks in the world."),
    Document(page_content="France is a country in Western Europe known for its rich history, culture, and cuisine."),
    
    # --- Tech / ML ---
    Document(page_content="Transformers are deep learning models that use attention mechanisms to process sequences."),
   
    # --- Space ---
    Document(page_content="Mars is known as the Red Planet due to its reddish appearance caused by iron oxide."),
    
    # --- Business / Customers ---
    Document(page_content="Customers who have not made a purchase in over 60 days are often considered at risk of churn."),
    
    # --- Mixed reasoning ---
    Document(page_content="Attention mechanisms allow models to focus on relevant parts of input data, improving performance in NLP tasks."),

]
# -------------------------
# 3. Chunking
# -------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
chunks = splitter.split_documents(documents)

texts = [c.page_content for c in chunks]

# -------------------------
# 4. Embeddings
# -------------------------
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedder.embed_documents(texts)
embeddings = np.array(embeddings).astype("float32")


# -------------------------
# 6. Query
# -------------------------
query = "When was Eiffel Tower built?"
q_emb = embedder.embed_query(query)
q_emb = np.array([q_emb]).astype("float32")


#euclidean distance between query and doc1
distances = np.linalg.norm(embeddings - q_emb, axis=1) 

print(f"length of distances: {len(distances)}") # 
print("Distances:", distances)





: 

In [41]:
similarity_matrix = embeddings@embeddings.T

In [43]:
# top 10 most values from similarity matrix and theier indices
top_k = 10
similarities = similarity_matrix[0]
top_k_indices = np.argsort(similarities)[-top_k:][::-1]


In [44]:
top_k_indices, similarities[top_k_indices]

(array([0, 2, 1, 6, 7, 3, 4, 5]),
 array([ 0.99999994,  0.29029936,  0.23485878,  0.06521597,  0.01519062,
         0.01223537, -0.04679701, -0.08668537], dtype=float32))